In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"), temperature=0.4)

c:\Users\harih\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sqlite3
from datetime import datetime, timezone

conn = sqlite3.connect("tasks.db", check_same_thread=False)
conn.row_factory = sqlite3.Row

conn.execute("""
CREATE TABLE IF NOT EXISTS tasks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    description TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'pending',
    due_date TEXT DEFAULT '',
    created_at TEXT NOT NULL,
    completed_at TEXT DEFAULT ''
)
""")
conn.commit()

def now():
    return datetime.now(timezone.utc).isoformat()

print("tasks table ready")

tasks table ready


In [5]:
from langchain_core.tools import tool

@tool
def add_task(description: str, due_date: str = "") -> str:
    """Add a new task with an optional due date (YYYY-MM-DD)."""
    cur = conn.execute(
        "INSERT INTO tasks (description, due_date, created_at) VALUES (?, ?, ?)",
        (description, due_date, now())
    )
    conn.commit()
    return f"Task added with id {cur.lastrowid}: '{description}'."

@tool
def list_pending_tasks() -> str:
    """List all pending (not completed) tasks."""
    rows = conn.execute(
        "SELECT id, description, due_date FROM tasks WHERE status = 'pending' ORDER BY created_at ASC"
    ).fetchall()
    if not rows:
        return "No pending tasks."
    return "\n".join(f"[{r['id']}] {r['description']} (due: {r['due_date'] or 'n/a'})" for r in rows)

@tool
def complete_task(task_id: int) -> str:
    """Mark a task as complete by id."""
    conn.execute("UPDATE tasks SET status = 'complete', completed_at = ? WHERE id = ?", (now(), task_id))
    conn.commit()
    return f"Task {task_id} marked complete."

@tool
def delete_task(task_id: int) -> str:
    """Delete a task by id."""
    conn.execute("DELETE FROM tasks WHERE id = ?", (task_id,))
    conn.commit()
    return f"Task {task_id} deleted."

@tool
def search_tasks(query: str) -> str:
    """Search tasks by keyword in the description. Matches singular/plural word forms."""
    words = [w.strip().lower() for w in query.split() if len(w.strip()) > 2]
    if not words:
        return "Please provide a more specific search term."
    variants = set()
    for w in words:
        variants.add(w)
        variants.add(w)
    if w.endswith("s"):
        variants.add(w[:-1])
    else:
        variants.add(w + "s")

    conditions = " OR ".join(["LOWER(description) LIKE ?"] * len(variants))
    params = [f"%{v}%" for v in variants]

    rows = conn.execute(f"SELECT id, description, status, due_date FROM tasks WHERE {conditions}", params).fetchall()
    if not rows:
        return f"No tasks matched '{query}'."
    return "\n".join(f"[{r['id']}] {r['description']} ({r['status']})" for r in rows)


task_toolkit = [add_task, list_pending_tasks, complete_task, delete_task, search_tasks]

In [6]:
print(add_task.invoke({"description": "Prepare tomorrow's workshop", "due_date": "2026-08-09"}))

Task added with id 1: 'Prepare tomorrow's workshop'.


In [7]:
from langchain.agents import create_agent
from datetime import date

today_str = date.today().strftime("%B %d, %Y")

system_prompt = f"""You are a task management assistant. Today's date is {today_str}.

You have tools to add, list, complete, delete, and search tasks.

Rules:
- Use the most specific tool for the request.
- Call each tool AT MOST ONCE per user request unless a follow-up action is clearly required.
- After using tools, give a clear, concise confirmation or summary."""

agent = create_agent(llm, tools=task_toolkit, system_prompt=system_prompt)

In [8]:
example_query = "Add a task to prepare the client presentation, due tomorrow."

events = agent.stream(
    {"messages": [("user", example_query)]},
    config={"recursion_limit": 8},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Add a task to prepare the client presentation, due tomorrow.
================================== Ai Message ==================================
Tool Calls:
  add_task (fc_bb087099-123c-4cde-9ccd-6a96cad773ec)
 Call ID: fc_bb087099-123c-4cde-9ccd-6a96cad773ec
  Args:
    description: Prepare the client presentation
    due_date: 2026-08-09
================================= Tool Message =================================
Name: add_task

Task added with id 2: 'Prepare the client presentation'.
================================== Ai Message ==================================

Task added (ID 2): “Prepare the client presentation”, due 2026‑08‑09. Let me know if you’d like to view, edit, or manage any other tasks.


In [9]:
example_query = "What are my pending tasks?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    config={"recursion_limit": 8},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What are my pending tasks?
================================== Ai Message ==================================
Tool Calls:
  list_pending_tasks (fc_4c05a762-f299-4815-86fd-649829ee0af3)
 Call ID: fc_4c05a762-f299-4815-86fd-649829ee0af3
  Args:
    : {}
================================= Tool Message =================================
Name: list_pending_tasks

[1] Prepare tomorrow's workshop (due: 2026-08-09)
[2] Prepare the client presentation (due: 2026-08-09)
================================== Ai Message ==================================

Here are your pending tasks:

1. **Prepare tomorrow's workshop** – due 2026‑08‑09  
2. **Prepare the client presentation** – due 2026‑08‑09
